# Homework 3 — Coding Project

This notebook covers **Section 4** of Homework 3:

- **Problem 17** (10 pts): Matrix factorization on synthetic panel data with SGD.
- **Problem 18** (5 pts): Self-attention and multi-head attention from scratch in PyTorch.

You'll need PyTorch 2.0+, NumPy, and matplotlib. CPU is fine — both tasks run in seconds.

Cells marked `TODO` are the ones you need to fill in. Other cells (setup, plotting, verification) are provided for you. Outputs are reproducible thanks to fixed seeds.

In [ ]:
# ============================================================================
# Imports and reproducibility
# ============================================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3)

print("Versions:")
print(f"  numpy:   {np.__version__}")
print(f"  torch:   {torch.__version__}")


---

## Part 1 — Matrix Factorization on Synthetic Panel Data (Problem 17)

We simulate a small panel dataset (think: $M = 20$ firms over $T = 12$ months of log-sales) generated from a known low-rank factor model:

$$
    Y_{ij} = \boldsymbol\lambda_i^\top \mathbf{F}_j + \varepsilon_{ij}, \qquad \boldsymbol\lambda_i \in \mathbb{R}^{K^*}, \mathbf{F}_j \in \mathbb{R}^{K^*}
$$

with **true rank $K^* = 3$**. Then we mask 25% of the entries (uniformly at random — think staggered treatment timing) and ask: can matrix factorization recover the held-out entries?

This setup mirrors **Athey, Bai, Imbens, Khosravi (2021)**: panel data with missing potential outcomes.

In [ ]:
# ============================================================================
# Generate synthetic panel data
# ============================================================================
M, T_periods, K_TRUE = 20, 12, 3   # 20 firms, 12 months, true rank 3
NOISE_SIGMA = 0.2
MASK_FRAC   = 0.25

rng = np.random.default_rng(42)

# True low-rank factors
Lambda_true = rng.standard_normal((M, K_TRUE)) * 0.7    # firm loadings
F_true      = rng.standard_normal((T_periods, K_TRUE)) * 0.7   # time factors

# Observed matrix = low-rank signal + Gaussian noise
Y = Lambda_true @ F_true.T + NOISE_SIGMA * rng.standard_normal((M, T_periods))

# Mask: 1 = observed, 0 = missing (held out)
mask = (rng.random((M, T_periods)) > MASK_FRAC).astype(np.float32)
print(f"Y shape:           {Y.shape}")
print(f"Observed fraction: {mask.mean():.3f}")
print(f"True rank:         {K_TRUE}")
print(f"Noise std:         {NOISE_SIGMA}")

# Build lists of (i, j, y_ij) for SGD over observed entries only
obs_idx = np.argwhere(mask == 1.0)
obs_y   = np.array([Y[i, j] for (i, j) in obs_idx], dtype=np.float32)
heldout = np.argwhere(mask == 0.0)
print(f"#observed:         {len(obs_idx)}")
print(f"#held out:         {len(heldout)}")


### Problem 17(a) — Implement MF with SGD

Implement `train_mf_sgd(...)` that returns the trained $\mathbf{U} \in \mathbb{R}^{M \times K}$ and $\mathbf{V} \in \mathbb{R}^{T \times K}$ by minimizing

$$
    L = \frac{\lambda}{2}\bigl(\|\mathbf{U}\|_F^2 + \|\mathbf{V}\|_F^2\bigr) + \frac{1}{2}\sum_{(i,j) \in S}(y_{ij} - \mathbf{u}_i^\top \mathbf{v}_j)^2.
$$

Use the per-observation gradients from **Problem 4**:

$$
    \frac{\partial L_{ij}}{\partial \mathbf{u}_i} = \lambda \mathbf{u}_i - (y_{ij} - \mathbf{u}_i^\top \mathbf{v}_j)\, \mathbf{v}_j, \qquad
    \frac{\partial L_{ij}}{\partial \mathbf{v}_j} = \lambda \mathbf{v}_j - (y_{ij} - \mathbf{u}_i^\top \mathbf{v}_j)\, \mathbf{u}_i.
$$

(Note: when implementing SGD, the regularization contribution gets spread across all observations — see the standard trick of dividing $\lambda$ by the number of observed entries per row/column, or use the simpler per-step regularizer $\lambda$ at each update.)

**Init:** $\mathbf{U}, \mathbf{V}$ uniform in $[-0.5, 0.5]$. **Learning rate:** $\eta = 0.03$.

In [ ]:
def train_mf_sgd(Y, mask, K, lam=0.0, eta=0.03, max_epochs=300, eps=1e-4, seed=0):
    \"\"\"
    Train matrix factorization via SGD on observed entries.

    Returns:
        U: (M, K) numpy array
        V: (T, K) numpy array
        history: list of in-sample losses (per epoch)
    \"\"\"
    M, T = Y.shape
    rng = np.random.default_rng(seed)
    U = rng.uniform(-0.5, 0.5, size=(M, K)).astype(np.float32)
    V = rng.uniform(-0.5, 0.5, size=(T, K)).astype(np.float32)

    obs = np.argwhere(mask == 1.0)
    n_obs = len(obs)
    history = []

    for epoch in range(max_epochs):
        # Shuffle observed indices each epoch
        perm = rng.permutation(n_obs)
        for idx in perm:
            i, j = obs[idx]
            e = Y[i, j] - U[i] @ V[j]
            # Gradient updates (spread regularizer per-step)
            grad_U = lam * U[i] - e * V[j]
            grad_V = lam * V[j] - e * U[i]
            U[i] = U[i] - eta * grad_U
            V[j] = V[j] - eta * grad_V

        # Compute in-sample loss
        recon = U @ V.T
        sq_err = 0.5 * ((mask * (Y - recon)) ** 2).sum()
        reg    = 0.5 * lam * ((U ** 2).sum() + (V ** 2).sum())
        loss = sq_err + reg
        history.append(loss)

        # Early stop
        if epoch >= 2:
            denom = history[0] - history[1] + 1e-12
            num   = history[-2] - history[-1]
            if num / denom < eps:
                break

    return U, V, history


# Sanity check: train with the true rank and look at the loss curve
U_hat, V_hat, hist = train_mf_sgd(Y, mask, K=K_TRUE, lam=0.0, eta=0.03)
print(f"K = {K_TRUE}: stopped after {len(hist)} epochs, final in-sample loss = {hist[-1]:.4f}")


### Problem 17(b) — Sweep $K$ and plot $E_\text{in}$, $E_\text{out}$

Train MF for $K \in \{1, 2, 3, 5, 8, 12\}$. Compute:

- $E_\text{in}$: squared loss on observed entries.
- $E_\text{out}$: squared loss on the held-out (masked) entries.

Plot both as a function of $K$ on the same axes.

In [ ]:
K_GRID = [1, 2, 3, 5, 8, 12]
ein_list, eout_list = [], []

for K in K_GRID:
    U_hat, V_hat, _ = train_mf_sgd(Y, mask, K=K, lam=0.0, eta=0.03, seed=K)
    recon = U_hat @ V_hat.T
    ein  = 0.5 * ((mask * (Y - recon)) ** 2).sum() / mask.sum()
    eout = 0.5 * (((1 - mask) * (Y - recon)) ** 2).sum() / (1 - mask).sum()
    ein_list.append(ein)
    eout_list.append(eout)
    print(f"K = {K:>2d}:  E_in = {ein:.3f}   E_out = {eout:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(K_GRID, ein_list,  'o-', label='E_in (observed)')
plt.plot(K_GRID, eout_list, 's-', label='E_out (held out)', color='red')
plt.axvline(K_TRUE, ls='--', color='gray', alpha=0.6, label=f'true K* = {K_TRUE}')
plt.xlabel('rank K')
plt.ylabel('average squared loss / 2')
plt.title('MF on synthetic panel: bias-variance vs K')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Problem 17(c) — Identify optimal $K$

Look at the $E_\text{out}$ curve. Which $K$ minimizes it? How close is it to the **true rank** ($K^* = 3$)?

In [ ]:
best_K = K_GRID[int(np.argmin(eout_list))]
print(f"Best K by E_out:  {best_K}")
print(f"True rank K*:     {K_TRUE}")
print(f"E_out at best K:  {min(eout_list):.4f}")


### Problem 17(d) — Regularization sweep

For each $\lambda \in \{0, 10^{-3}, 10^{-2}, 10^{-1}\}$, sweep $K \in \{1, 2, 3, 5, 8, 12\}$ and plot $E_\text{out}$ vs $K$. What changes as $\lambda$ grows?

In [ ]:
LAM_GRID = [0, 1e-3, 1e-2, 1e-1]

plt.figure(figsize=(7, 4.5))
for lam in LAM_GRID:
    eout_lam = []
    for K in K_GRID:
        U_hat, V_hat, _ = train_mf_sgd(Y, mask, K=K, lam=lam, eta=0.03, seed=K)
        recon = U_hat @ V_hat.T
        eout = 0.5 * (((1 - mask) * (Y - recon)) ** 2).sum() / (1 - mask).sum()
        eout_lam.append(eout)
    plt.plot(K_GRID, eout_lam, 'o-', label=f'$\\lambda = {lam}$')

plt.axvline(K_TRUE, ls='--', color='gray', alpha=0.5)
plt.xlabel('rank K');  plt.ylabel('E_out')
plt.title('E_out vs K for several regularization strengths')
plt.legend();  plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("\nObservation: with no regularization, E_out blows up for K > K*.")
print("With moderate lambda, even large K stays reasonable -- the regularizer")
print("'soft-shrinks' unused dimensions toward zero. Classic bias-variance.")


---

## Part 2 — Self-Attention from Scratch (Problem 18)

You will implement scaled dot-product self-attention and multi-head attention as PyTorch modules, **without** using `nn.MultiheadAttention` or `F.scaled_dot_product_attention`. Then verify your implementation matches PyTorch's built-in to numerical tolerance.

In [ ]:
# Setup: a small random input
T_seq, D = 7, 16   # 7 tokens of embedding dim 16
x = torch.randn(T_seq, D)
print(f"input shape: {tuple(x.shape)}")


### Problem 18(a) — Single-head scaled dot-product self-attention

Fill in `SelfAttention` below. The forward pass should compute

$$
    \mathbf{Q} = \mathbf{x}\mathbf{W}_Q^\top, \quad
    \mathbf{K} = \mathbf{x}\mathbf{W}_K^\top, \quad
    \mathbf{V} = \mathbf{x}\mathbf{W}_V^\top,
    \quad
    \text{Attention} = \operatorname{softmax}\!\Bigl(\tfrac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\Bigr) \mathbf{V}.
$$

Use `nn.Linear` for the three projections (it stores weight as `(out, in)` so `Linear(D, d_k)(x)` gives `x @ W.T`). Bias is optional — use `bias=False` for simplicity.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, D, d_k, d_v):
        super().__init__()
        self.W_Q = nn.Linear(D, d_k, bias=False)
        self.W_K = nn.Linear(D, d_k, bias=False)
        self.W_V = nn.Linear(D, d_v, bias=False)
        self.d_k = d_k

    def forward(self, x):
        # x: (T, D)
        Q = self.W_Q(x)        # (T, d_k)
        K = self.W_K(x)        # (T, d_k)
        V = self.W_V(x)        # (T, d_v)

        scores  = (Q @ K.T) / (self.d_k ** 0.5)     # (T, T)
        weights = F.softmax(scores, dim=-1)         # row-wise softmax
        out     = weights @ V                       # (T, d_v)
        return out, weights


# Build and test
torch.manual_seed(0)
sa = SelfAttention(D=D, d_k=D, d_v=D)
out, attn = sa(x)
print(f"out shape:   {tuple(out.shape)}")     # (7, 16)
print(f"attn shape:  {tuple(attn.shape)}")    # (7, 7)
print(f"attn rows sum to 1: {torch.allclose(attn.sum(dim=-1), torch.ones(T_seq), atol=1e-6)}")


### Problem 18(a) verification — match PyTorch's built-in

Reuse the same $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ projections from your module and compare with `torch.nn.functional.scaled_dot_product_attention`. Tolerance: $10^{-5}$.

In [ ]:
# Cross-check: use our projections, then call PyTorch's built-in
with torch.no_grad():
    Q = sa.W_Q(x).unsqueeze(0)   # add a batch dim for F.scaled_dot_product_attention
    K = sa.W_K(x).unsqueeze(0)
    V = sa.W_V(x).unsqueeze(0)

torch_out = F.scaled_dot_product_attention(Q, K, V).squeeze(0)
our_out, _ = sa(x)

diff = (our_out - torch_out).abs().max().item()
print(f"Max abs diff vs F.scaled_dot_product_attention: {diff:.2e}")
print(f"Match (< 1e-5): {diff < 1e-5}")


### Problem 18(b) — Multi-head attention with $\mathbf{W}_O$

Fill in `MultiHeadAttention`. With $h$ heads and total output dim $D$, the standard convention is $d_k = d_v = D / h$ so each head sees a sub-space.

For each head $i$:

$$
    \mathbf{O}^i = \operatorname{Attention}(\mathbf{Q}^i, \mathbf{K}^i, \mathbf{V}^i) \in \mathbb{R}^{T \times d_v}.
$$

Concatenate and project:

$$
    \mathbf{O} = [\mathbf{O}^1 \;\mathbf{O}^2\; \cdots\; \mathbf{O}^h]\, \mathbf{W}_O^\top \in \mathbb{R}^{T \times D}.
$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, D, h):
        super().__init__()
        assert D % h == 0, "D must be divisible by h"
        self.h = h
        self.d_k = D // h
        self.heads = nn.ModuleList([
            SelfAttention(D=D, d_k=self.d_k, d_v=self.d_k) for _ in range(h)
        ])
        self.W_O = nn.Linear(D, D, bias=False)

    def forward(self, x):
        head_outs = [head(x)[0] for head in self.heads]   # each (T, d_v)
        cat = torch.cat(head_outs, dim=-1)                 # (T, h * d_v) = (T, D)
        return self.W_O(cat)


# Test
torch.manual_seed(0)
mha = MultiHeadAttention(D=16, h=4)
out_mha = mha(x)
print(f"input shape:   {tuple(x.shape)}")
print(f"output shape:  {tuple(out_mha.shape)}")
print(f"shape preserved: {x.shape == out_mha.shape}")


---

## Part 3 — Word2Vec on dr_seuss.txt (Problem 19)

Now we implement the **skip-gram model** end-to-end on a small text corpus and inspect the learned embeddings. We use the same `dr_seuss.txt` dataset that the Caltech CS155 set-5 problem uses (2,071 words, 308 unique).

Pipeline:

1. Tokenize the text and build a `word_to_index` dictionary.
2. Generate `(input_word, context_word)` training pairs with a sliding window.
3. Train a 2-layer neural network: `Linear(W, D) → Linear(D, W)` with cross-entropy loss.
4. The hidden-layer weight matrix gives the word embeddings (one row per word).
5. Compute cosine similarity between every pair of words; report the top-30 most-similar pairs.

This is the practical embodiment of the skip-gram math from **Problem 7** and **Problem 8**.

In [ ]:
# ============================================================================
# Helper functions (adapted from Caltech CS155 P3CHelpers.py)
# ============================================================================
import os

class WordPair:
    \"\"\"Pair of words plus a similarity score.\"\"\"
    def __init__(self, w1, w2, sim):
        assert w1 != w2
        self.w1, self.w2, self.sim = w1, w2, sim
    def __repr__(self):
        return f"({self.w1}, {self.w2})  sim = {self.sim:.4f}"

def load_word_list(path):
    \"\"\"Strip non-alphanumeric chars, lowercase, return token list.\"\"\"
    with open(path) as f:
        raw = f.read().strip().split()
    tokens = []
    for w in raw:
        w = ''.join(c for c in w if c.isalnum()).lower()
        if w:
            tokens.append(w)
    return tokens

def build_vocab(token_list):
    \"\"\"Return dict mapping each unique word to its index (insertion order).\"\"\"
    w2i = {}
    for w in token_list:
        if w not in w2i:
            w2i[w] = len(w2i)
    return w2i

def cosine_sim(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

# Load corpus
data_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'data', 'dr_seuss.txt')
if not os.path.exists(data_path):
    data_path = 'data/dr_seuss.txt'   # fallback for Colab / different working dirs
tokens = load_word_list(data_path)
w2i = build_vocab(tokens)
print(f"Tokens loaded: {len(tokens)}")
print(f"Vocabulary size W = {len(w2i)}")
print(f"First 10 tokens:  {tokens[:10]}")


### Problem 19(a) — Generate training pairs

Implement `generate_traindata(tokens, w2i, window_size=4)`.

For each position $t$, emit a `(center, context)` pair for every offset $j$ with $|j| \leq s$ and $j \neq 0$ (and within bounds). Both center and context should be one-hot encoded.

Return `trainX, trainY` as NumPy arrays of shape `(N_pairs, W)`.

In [ ]:
def one_hot(w2i, word):
    v = np.zeros(len(w2i), dtype=np.float32)
    v[w2i[word]] = 1.0
    return v

def generate_traindata(tokens, w2i, window_size=4):
    trainX, trainY = [], []
    n = len(tokens)
    for i in range(n):
        for j in range(max(0, i - window_size), min(n, i + window_size + 1)):
            if j != i:
                trainX.append(one_hot(w2i, tokens[i]))
                trainY.append(one_hot(w2i, tokens[j]))
    return np.array(trainX), np.array(trainY)


trainX, trainY = generate_traindata(tokens, w2i, window_size=4)
print(f"trainX shape: {trainX.shape}    # (N_pairs, W)")
print(f"trainY shape: {trainY.shape}")
print(f"Total training pairs: {len(trainX)}")


### Problem 19(b) — Train the skip-gram network and extract embeddings

Build a 2-layer linear network:
- Hidden: `nn.Linear(W, D)` (no activation) — its weight matrix \emph{is} the embedding.
- Output: `nn.Linear(D, W)` followed by softmax (handled implicitly by `CrossEntropyLoss`).

Train for ~500 epochs (or until loss plateaus). Use Adam with lr = 0.01. Then extract the hidden-layer weight matrix as the embedding.

In [ ]:
D_EMBED = 10
W = len(w2i)

torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(W, D_EMBED),       # hidden layer = the embeddings
    nn.Linear(D_EMBED, W),       # output projection
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

trainX_t = torch.tensor(trainX, dtype=torch.float32)
trainY_t = torch.tensor(np.argmax(trainY, axis=1), dtype=torch.long)

for epoch in range(500):
    optimizer.zero_grad()
    logits = model(trainX_t)
    loss = criterion(logits, trainY_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"  epoch {epoch+1:>3d}   loss = {loss.item():.4f}")

# nn.Linear weight is (out, in). For one row per word, transpose.
embeddings = model[0].weight.data.numpy().T   # shape (W, D_EMBED)
print(f"\nEmbedding matrix shape: {embeddings.shape}")


### Problem 19(c) — Top 30 most similar word pairs

For each word, find its nearest neighbor by cosine similarity. Print the top 30 highest-similarity pairs.

In [ ]:
def top_similar_pairs(embeddings, w2i, top_n=30):
    idx2w = {i: w for w, i in w2i.items()}
    W = len(w2i)
    # Pre-normalize all rows
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / np.where(norms > 0, norms, 1.0)
    sim = normed @ normed.T                # (W, W)
    np.fill_diagonal(sim, -np.inf)         # exclude self

    pairs = []
    for i in range(W):
        j = int(np.argmax(sim[i]))
        pairs.append(WordPair(idx2w[i], idx2w[j], float(sim[i, j])))

    # Sort by similarity (descending), then deduplicate (a,b) vs (b,a)
    pairs = sorted(pairs, key=lambda p: -p.sim)
    seen = set()
    unique = []
    for p in pairs:
        key = tuple(sorted([p.w1, p.w2]))
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique

pairs = top_similar_pairs(embeddings, w2i)
print(f"Top 30 most-similar word pairs:")
for p in pairs[:30]:
    print(f"  {p}")


### Problem 19(d) — Reflection

Look at the top-30 pairs above and answer (briefly, in the markdown cell of your write-up):

1. What dimensions are the hidden-layer weight matrix and the output-layer weight matrix?
2. Do the resulting pairs make linguistic / semantic sense, or are they mostly noise? Why might that be, given the corpus is only a few thousand words?
3. The Caltech write-up notes that many pairs look like **rhymes** (e.g., `wump` / `hump`, `goat` / `boat`). Why? *Hint:* Dr.\ Seuss writes in rhyming verse, so rhyming words tend to appear in similar contexts.

---

## Part 4 — Applied Transformer on Real Text (Problem 20)

So far we've built attention modules from scratch. Now we use a **real, pretrained Transformer** on real text and inspect what attention has learned.

We'll use `distilbert-base-uncased-finetuned-sst-2-english` from HuggingFace -- a small (66M-parameter) Transformer fine-tuned for binary sentiment classification on the SST-2 dataset.

You'll:
1. Load the model and tokenize a sentence.
2. Run a forward pass and read the sentiment prediction.
3. Pull out the **attention weights from one layer / head** and visualize the attention map.
4. Apply to a finance-relevant sentence and discuss what the model attends to.

**Setup**: the cell below installs `transformers` if needed. First-time run downloads ~250 MB.

In [ ]:
# ============================================================================
# Install transformers if not present
# ============================================================================
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print("transformers available.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "transformers"])
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print("transformers installed.")


In [ ]:
# Load pretrained model + tokenizer
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_attentions=True)
model_bert.eval()

print(f"Loaded {MODEL_NAME}")
print(f"  vocab size: {tokenizer.vocab_size}")
print(f"  layers:     {model_bert.config.n_layers}")
print(f"  heads/lyr:  {model_bert.config.n_heads}")
print(f"  hidden dim: {model_bert.config.dim}")


### Problem 20(a) — Sentiment of finance-flavored sentences

Run inference on several finance sentences. Report whether the prediction is POSITIVE or NEGATIVE and the model's confidence.

In [ ]:
SENTENCES = [
    "The Federal Reserve raised interest rates to curb inflation.",
    "Earnings exceeded analyst expectations by a wide margin.",
    "Unemployment rose sharply and consumer confidence plummeted.",
    "The bank announced a record dividend and an aggressive buyback program.",
    "Inflation expectations have de-anchored and risks remain to the upside.",
]

with torch.no_grad():
    for sent in SENTENCES:
        inputs = tokenizer(sent, return_tensors="pt")
        out = model_bert(**inputs)
        probs = torch.softmax(out.logits, dim=-1).squeeze(0).tolist()
        label = "POSITIVE" if probs[1] > probs[0] else "NEGATIVE"
        conf = max(probs)
        print(f"  {label:>8}  ({conf:.3f})  {sent}")


### Problem 20(b) — Extract and visualize an attention map

Pick the **finance sentence of your choice**. Tokenize it, run inference, and pull out the attention tensor `outputs.attentions` which has shape `(num_layers, batch, num_heads, T, T)`.

Plot the attention map for **layer 3, head 0** (i.e., `attentions[3][0, 0]`). The horizontal axis is "key" (what is being attended to); the vertical axis is "query" (what is doing the attending).

In [ ]:
SENT = "Earnings exceeded analyst expectations by a wide margin."

inputs = tokenizer(SENT, return_tensors="pt")
with torch.no_grad():
    out = model_bert(**inputs)

attn_map = out.attentions[3][0, 0].numpy()       # layer 3, head 0
tokens_bert = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens_bert)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_map, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(tokens_bert)))
ax.set_xticklabels(tokens_bert, rotation=45, ha='right')
ax.set_yticks(range(len(tokens_bert)))
ax.set_yticklabels(tokens_bert)
ax.set_xlabel('key (attended to)')
ax.set_ylabel('query (attending)')
ax.set_title('DistilBERT attention -- layer 3, head 0')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


### Problem 20(c) — Reflection

In a short markdown cell, answer:

1. Was the model's sentiment prediction for your sentence consistent with what a human would say?
2. Looking at the attention map, which tokens attend most strongly to which? Do the patterns look meaningful (e.g., adjectives attending to the nouns they modify, the `[CLS]` token attending to sentiment-bearing words)?
3. Try the same exercise with a sentence where the sentiment is **ambiguous** or **mixed** (e.g., "The hike was widely expected but the press conference was hawkish"). Does the model still produce a confident label? What does the attention map look like?

The point of this exercise: you've gone from \emph{deriving} attention math to \emph{interpreting} what attention does in a real, deployed model.

---

### Final check

Quick sanity for all four parts:
- **Part 1 (MF):** $K^*$ from $E_\text{out}$ is close to the true rank ($K^* = 3$).
- **Part 2 (Self-attention):** output matches `F.scaled_dot_product_attention` to $10^{-5}$.
- **Part 3 (Word2vec):** top-similar pairs include several rhymes (the Dr.\ Seuss artifact).
- **Part 4 (Applied Transformer):** attention maps are interpretable on real sentences.

Save the notebook and submit it together with the PDF math write-up.

In [ ]:
print("Homework 3 coding complete.")
print(f"  Part 1 -- MF: best K by E_out = {best_K} (true K* = {K_TRUE})")
print(f"  Part 2 -- MHA preserves shape: {x.shape == mha(x).shape}")
print(f"  Part 3 -- Word2vec embeddings: {embeddings.shape}")
print(f"  Part 4 -- DistilBERT loaded with {model_bert.config.n_layers} layers and {model_bert.config.n_heads} heads.")
